# M03.01 — RAG: the base case

A retrieval pipeline over the six Groupe Mutuel terms and conditions documents in `data/insurance_terms/`: embeddings, cosine similarity, a small vector store, chunking, retrieval and generation, all in plain NumPy.

1. Why retrieve at all?
2. Embeddings and cosine similarity
3. A minimal vector store
4. Chunking and indexing
5. Retrieval
6. Generation

## Setup

In [ ]:
import inspect

import numpy as np
from openai.types.responses import Response

from agentic_workshop.client import get_client
from agentic_workshop.config import CHAT_MODEL, EMBEDDING_MODEL
from agentic_workshop.costs import PRICES, print_cost, usage_cost
from agentic_workshop.documents import POLICY_DOCUMENTS, load_pdf_text
from agentic_workshop.vector_store import MiniVectorStore

client = get_client()
MODEL = CHAT_MODEL

## 1. Why retrieve at all?

### 1.1 No context

In [ ]:
question = (
    "Under Groupe Mutuel's Optimum supplemental insurance, what share of psychotherapy costs is reimbursed, and up to what annual maximum?"
)

response = client.responses.create(model=MODEL, input=question)

print("LLM response:")
print(response.output_text)
print()
print_cost(response.usage)

But according to `data/insurance_terms/SOGA01-E1.pdf`, the correct answer is 90%, up to CHF 500 per year.

### 1.2 The whole corpus in the prompt

In [ ]:
# One text per PDF, keyed by document code, e.g. "DPAM01-E4" (agentic_workshop/documents.py reads them with pypdf).
documents = {doc_code: load_pdf_text(doc_code) for doc_code in POLICY_DOCUMENTS}

for name, text in documents.items():
    print(f"{name:10s} {len(text):6d} characters")

full_corpus = "\n\n".join(documents.values())
print(f"\nFull corpus: {len(full_corpus)} characters, roughly {len(full_corpus) // 4} tokens")

In [ ]:
prompt = f"""Answer the question using only the policy documents below.

POLICY DOCUMENTS
{full_corpus}

QUESTION
{question}
"""

response_full = client.responses.create(model=MODEL, input=prompt)

print(response_full.output_text)
print()
print_cost(response_full.usage)
cost_full = usage_cost(response_full.usage)

This works because the corpus is small (about 20k tokens). Compare the cost with the call in 1.1: even a few documents multiply it. As the corpus grows, cost, latency and accuracy all get worse (Extension 1).

## 2. Embeddings

An embedding model turns a text into a fixed-length vector, such that texts with similar meanings get vectors pointing in similar directions.

In [ ]:
def embed(texts: list[str]) -> list[np.ndarray]:
    """Embed a list of texts. Returns one vector per input, preserving order."""

    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    vectors = [np.array(item.embedding) for item in response.data]

    tokens = response.usage.total_tokens
    print(f"Embedded {len(texts)} text(s), {tokens} tokens, "
          f"${tokens / 1e6 * PRICES[EMBEDDING_MODEL]['input']:.6f}")

    return vectors


vector = embed(["Psychotherapy must be prescribed by a doctor."])[0]

print(f"\nDimensions: {vector.shape[0]}")
print(f"First 8 values: {np.round(vector[:8], 4)}")
print(f"L2 norm: {np.linalg.norm(vector):.4f}")

The vectors are already unit length (see Extension 3).

### 2.1 Cosine similarity

Cosine similarity measures whether two vectors point in the same direction:

$$\text{cosine}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

For unit-length vectors, the denominator is 1 and this is just the dot product.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine of the angle between two vectors."""

    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

Let's compare text that's semantically related to the question vs unrelated text.

In [ ]:
query = "How do I get my money back after paying a doctor?"

# Three example sentences adapted from the corpus.
# The fourth is completely unrelated.
candidates = [
    "The insurer can only settle accounts based on detailed invoices indicating the treatment dates, type of treatment and the cost of each benefit.",
    "As a rule, insured persons are responsible for covering fees directly to healthcare providers.",
    "DP insurance coverage is open to all persons under 60.",
    "The Matterhorn is located in the canton of Valais.",
]

query_vector = embed([query])[0]
candidate_vectors = embed(candidates)

print()
for text, vec in zip(candidates, candidate_vectors, strict=True):
    print(f"{cosine_similarity(query_vector, vec):.4f}  {text}")

- Awesome: the payment sentences score highest without sharing words with the query.
- But keep in mind: scores are relative, and not calibrated. There is no universal relevance threshold that works across all queries.

## 3. A minimal vector store

A vector store keeps vectors, their texts and metadata, and returns the entries closest to a query vector. Ours does an exhaustive search with the cosine similarity above; production stores add persistence, indexes for approximate search and richer filtering.

The class we use for the rest of the day lives in `agentic_workshop/vector_store.py`; open it there to read or change it. Here is its code:

In [ ]:
print(inspect.getsource(MiniVectorStore))

## 4. Chunking and indexing

One vector for a whole document is too diluted, so we split each document into overlapping fixed-size windows. M03.02 shows where this breaks.

In [ ]:
def chunk_text(text: str, chunk_size: int = 600, overlap: int = 100, separator: str | None = None) -> list[str]:
    """Split text into overlapping fixed-size character windows, or, given a separator, at every occurrence of it."""

    if separator is not None:
        # Keep the separator with the piece it starts, e.g. "\nArt. " gives pieces starting "Art. 3 Eligibility".
        pieces = text.split(separator)
        pieces = pieces[:1] + [separator + p for p in pieces[1:]]
        return [p.strip() for p in pieces if p.strip()]

    if not 0 <= overlap < chunk_size:
        raise ValueError(f"overlap must be at least 0 and smaller than chunk_size, got overlap={overlap}, chunk_size={chunk_size}.")
    step = chunk_size - overlap
    chunks = []

    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break

    return chunks

In [ ]:
# Chunk each document separately, so a chunk never spans two documents.
chunks = []
for name, text in documents.items():
    for position, chunk in enumerate(chunk_text(text)):
        chunks.append({"text": chunk, "source": name, "position": position})

print(f"{len(documents)} documents -> {len(chunks)} chunks\n")

for name in documents:
    n = sum(1 for c in chunks if c["source"] == name)
    print(f"{name:10s} {n} chunks")

In [ ]:
# Look at one chunk to see what the embedding model is actually being given.
example = next(c for c in chunks if c["source"] == "SOGA01-E1" and "Psychotherapy" in c["text"])
print(f"source: {example['source']}  position: {example['position']}\n")
print(repr(example["text"]))

This is what the embedding model sees: a flattened table and arbitrary boundaries. M03.02 compares chunking strategies that do better.

In [ ]:
store = MiniVectorStore()

vectors = embed([c["text"] for c in chunks])
store.add(
    embeddings=vectors,
    texts=[c["text"] for c in chunks],
    metadata=[{"source": c["source"], "position": c["position"]} for c in chunks],
)

print(f"\nIndexed {len(store)} chunks.")

Indexing is cheap compared to querying, and usually only needs to be done once.

## 5. Retrieval

In [ ]:
def retrieve(question: str, k: int = 3) -> list[dict]:
    """Return the k chunks most similar to the question."""

    question_vector = embed([question])[0]
    return store.query(question_vector, k=k)


def show(results: list[dict]) -> None:
    for rank, r in enumerate(results, start=1):
        preview = " ".join(r["text"].split())[:180]
        print(f"\n[{rank}] {r['score']:.4f}  ({r['metadata']['source']}, "
              f"chunk {r['metadata']['position']})")
        print(f"    {preview}...")

In [ ]:
results = retrieve(question, k=3)

print(f"Question: {question}")
show(results)

Try other values of `k`, and rephrase the question.

## 6. Generation

In [ ]:
def format_context(results: list[dict]) -> str:
    """Turn retrieved chunks into a labelled block for the prompt."""

    return "\n\n".join(
        f"[{r['metadata']['source']}]\n{r['text']}" for r in results
    )


def answer(question: str, k: int = 3, verbose: bool = True) -> tuple[Response, list[dict]]:
    results = retrieve(question, k=k)

    prompt = f"""Answer the question using only the policy extracts below.
Cite the source document for every fact you state.

POLICY EXTRACTS
{format_context(results)}

QUESTION
{question}
"""

    response = client.responses.create(model=MODEL, input=prompt)

    if verbose:
        print(f"Q: {question}\n")
        print(response.output_text)
        print()
        print_cost(response.usage)

    return response, results

In [ ]:
response_rag, _ = answer(question, k=3)

In [ ]:
for q in [
    "How much does Optimum pay towards glasses for an adult?",
    "Is cosmetic teeth whitening covered?",
    "Does my supplemental insurance cover dental treatment?",
]:
    _, hits = answer(q, k=3)
    print("retrieved:", [h["metadata"]["source"] for h in hits])
    print("\n" + "=" * 80 + "\n")

Check each answer against the retrieved sources:

- **Glasses:** one table row in one chunk, so it works.
- **Teeth whitening:** the deciding exclusion (`CAGA04-E2` Art. 18, "cosmetic imperfections") shares no words with the question and is not retrieved.
- **Dental treatment:** needs three documents (the product, Dentaire Plus and the General Conditions). Look at the `retrieved:` line: how many of them were found?

### 6.1 What retrieval saved

In [ ]:
cost_rag = usage_cost(response_rag.usage)

print(f"Whole corpus in the prompt: {cost_full['input_tokens']:6d} input tokens, "
      f"${cost_full['cost_usd']:.6f}")
print(f"Top-3 chunks in the prompt: {cost_rag['input_tokens']:6d} input tokens, "
      f"${cost_rag['cost_usd']:.6f}")
print(f"\nReduction: {100 * (1 - cost_rag['input_tokens'] / cost_full['input_tokens']):.0f}% "
      f"fewer input tokens")

Stuffing grows with the corpus; retrieval grows with `k`.

### 6.2 A question the corpus does not answer

In [ ]:
unanswerable = "Does Groupe Mutuel cover LASIK laser eye surgery? Answer only yes/no"

show(retrieve(unanswerable, k=3))

In [ ]:
response_bad, _ = answer(unanswerable, k=3)

- Look at what was retrieved: page footers that mention "Groupe Mutuel".
- `query` always returns `k` results, relevant or not (Extension 4).

## Takeaways

- Without context, the model answers anyway. You can't tell whether the answer is invented or what it's based on.
- A vector store keeps track of vectors and their source chunks.
- Embeddings match meaning, not vocabulary. This is a strength, but also a failure mode.
- Similarity scores are not calibrated: the relative order matters, the exact values don't compare across queries.
- The retriever always returns `k` chunks, relevant or not. For an unanswerable question, it can return page footers for example.
- If the corpus fits in the prompt (about 20k tokens here), you may not need RAG. If typical questions need reasoning across documents, you may need more than RAG (e.g. agentic RAG, M04).

In [ ]:
# If you have been inspecting the retrieved chunks, you'll see that they often look a bit off.
# Fixed-size windows cut articles and table rows in half. We'll cover this in-depth in M03.02
# For the extensions, we recommend to re-index with one chunk per article instead:
chunks = []
for name, text in documents.items():
    for position, chunk in enumerate(chunk_text(text, separator="\nArt. ")):
        chunks.append({"text": chunk, "source": name, "position": position})

store = MiniVectorStore()
store.add(
    embeddings=embed([c["text"] for c in chunks]),
    texts=[c["text"] for c in chunks],
    metadata=[{"source": c["source"], "position": c["position"]} for c in chunks],
)
lengths = [len(c["text"]) for c in chunks]
print(f"\n{len(documents)} documents -> {len(store)} article chunks, "
      f"{min(lengths)} to {max(lengths)} characters (median {int(np.median(lengths))})")

## Extensions

**Easy**

1. **Cost at scale.** Assume 400k tokens of terms and 10'000 questions a month. Compare the monthly cost of stuffing the entire corpus in the context against top-5 retrieval, using the notebook's token counts. Assume a chunk size of 600, with about 4 characters per token. How are latency and quality affected?
2. **Query rewriting.** Have the model restate the question in policy vocabulary (give examples) and retrieve with the rephrased query. Try *"my dentist billed me 800 francs to whiten my teeth, now what?"*, and other somewhat ambiguously phrased questions. You can also have the model split multi-part questions into sub-queries and retrieve for each.

**Medium**

3. **Whitened cosine.** For unit embedding vectors, cosine, dot product and Euclidean distance give the same ranking (why?). Changing the geometry can affect ranking: embedding spaces are often anisotropic, i.e. with every vector in a narrow cone. Centre the embeddings on the corpus mean and apply PCA. Then rank with cosine. Does it improve retrieval on a few questions whose answering article you know?
4. **Knowing when there is nothing to find.** Can you find a score threshold that separates answerable from unanswerable questions? Test it on new questions. Compare it with a prompt that instructs the LLM to say a question was "not covered" by the retrieved documents.

**Hard**

5. **Hybrid retrieval.** Implement BM25, find queries where it beats dense retrieval (codes, abbreviations) and where it loses. Combine both dense (RAG) and sparse (BM25) search with Reciprocal Rank Fusion (each item becomes re-scored as the sum of `1 / (60 + rank)` for each individual ranking, BM25 or dense).

In [ ]:
# Extension 1: cost at scale
corpus_tokens, questions_per_month, k = 400000, 10000, 5
price = PRICES[MODEL]

chunk_tokens = 600 / 4  # chunk_size in characters, let's say about 4 characters per token as we do in 1.2
overhead = cost_rag["input_tokens"] - 3 * chunk_tokens  # instructions + question
output_tokens = cost_rag["output_tokens"]

monthly_cost = {}
scenarios = {
    "Stuffing": (corpus_tokens + overhead, price["input"]),
    "Stuffing, corpus cached": (corpus_tokens + overhead, price["cached_input"]),
    f"Top-{k} retrieval": (k * chunk_tokens + overhead, price["input"]),
}
for name, (tokens, input_price) in scenarios.items():
    monthly = monthly_cost[name] = questions_per_month * (tokens * input_price + output_tokens * price["output"]) / 1e6
    print(f"{name:24s} {tokens:9,.0f} input tokens/question  ${monthly:10,.2f}/month")

embedding_price = PRICES[EMBEDDING_MODEL]["input"]
print(f"\nRetrieval also pays for indexing once: ${corpus_tokens * embedding_price / 1e6:.4f}")

# Caching works because the corpus comes before the question, so every prompt shares that prefix.
# Latency: time to first token grows with the input, so 400k tokens per question is slow (and may not
# fit the context window at all). Quality: facts in the middle of a long context are easily missed,
# while retrieval only fails when the right chunk is not in the top k.

In [ ]:
# Extension 2: query rewriting
def rewrite(question: str) -> tuple[list[str], Response]:
    """Restate a question in policy vocabulary, one search query per line (several if it has several parts)."""

    prompt = f"""Rewrite the customer question as search queries over Swiss health insurance terms and conditions.
Use the vocabulary of the policy documents, not the customer's. Write one query per part of the question.
Output only the queries, one per line.

Examples:
"can I get my money back for my glasses?" -> Reimbursement of spectacles and contact lenses
"my kid needs braces, do you pay?" -> Orthodontic treatment for children

QUESTION
{question}"""
    response = client.responses.create(model=MODEL, input=prompt)
    return [line.strip() for line in response.output_text.splitlines() if line.strip()], response

# Try out different ones!
# messy = "my dentist billed me 800 francs to whiten my teeth, now what?"
# messy = "I'm going on holiday, will I be covered?"
messy = "I chipped my tooth while eating food that had a small pebble in it, what do I do?"
print(f"Original: {messy}")
show(retrieve(messy, k=3))

queries, rewrite_response = rewrite(messy)
for q in queries:
    print(f"\nRewritten: {q}")
    show(retrieve(q, k=3))

# Are the retrieved chunks after reframing consistently more relevant?
# Remember that misses can also be explained by improper chunking!

print("\nCost of the rewrite:")
print_cost(rewrite_response.usage)

# At the scale of Extension 1: one extra LLM call per question, and every extra sub-query adds k chunks
# to the answer prompt. Embedding the rewritten queries is negligible.
rewrite_cost = usage_cost(rewrite_response.usage)["cost_usd"]
extra_chunks_cost = (len(queries) - 1) * k * chunk_tokens * price["input"] / 1e6
top_k = monthly_cost[f"Top-{k} retrieval"]
with_rewrite = top_k + questions_per_month * (rewrite_cost + extra_chunks_cost)
print(f"\nTop-{k} retrieval:                ${top_k:8,.2f}/month")
print(f"Top-{k} retrieval + rewriting:    ${with_rewrite:8,.2f}/month ({len(queries)} sub-quer{'y' if len(queries) == 1 else 'ies'})")

# Most of the rewrite's cost is reasoning tokens: a rewrite needs none, so pass reasoning=REASONING_OFF
# (agentic_workshop.config) to make it cheaper and faster.

In [ ]:
# Extension 3: whitened cosine ("all-but-the-top")
# For unit vectors, ||a - b||² = 2 - 2 a·b and cosine = a·b: all three metrics rank identically.
E = np.stack(store.embeddings)
mean = E.mean(axis=0)
_, singular_values, components = np.linalg.svd(E - mean, full_matrices=False)  # rows: principal directions


def mean_cosine(X: np.ndarray) -> float:
    """Average cosine similarity between two different rows of X."""

    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    similarities = X @ X.T
    return float(similarities[~np.eye(len(X), dtype=bool)].mean())


variance = singular_values**2 / (singular_values**2).sum()
print(f"Mean cosine between two different chunks: {mean_cosine(E):.3f} raw, {mean_cosine(E - mean):.3f} centred")
print(f"Share of variance in the top 5 components: {np.round(variance[:5], 3)}\n")


def transform(X: np.ndarray, remove: int = 0, whiten: int = 0) -> np.ndarray:
    """Centre on the corpus mean, then drop the top `remove` components, or keep and whiten the top `whiten`."""

    X = X - mean
    if remove:
        top = components[:remove]
        X = X - (X @ top.T) @ top
    if whiten:
        X = (X @ components[:whiten].T) / singular_values[:whiten]
    return X


# A small test set: each question with the article(s) that answer it, as (source, start of the chunk).
labelled = {
    question: [("SOGA01-E1", "Art. 5 ")],
    "How much does Optimum pay towards glasses for an adult?": [("SOGA01-E1", "Art. 5 ")],
    "Is cosmetic teeth whitening covered?": [("CAGA04-E2", "Art. 18 ")],
    "How do I get my money back after paying a doctor?": [("CAGA04-E2", "Art. 17 ")],
    "Am I covered when I travel abroad?": [("CAGA04-E2", "Art. 4 ")],
    "How long do I have to make a claim?": [("CAGA04-E2", "Art. 38 ")],
    "Can I cancel my supplemental insurance?": [("CAGA04-E2", "Art. 13 ")],
    "Is there a discount if my children are insured too?": [("SOGA01-E1", "Art. 9 ")],
    "Does Dentaire Plus pay for a yearly check-up at the dentist?": [("DPAM01-E4", "Art. 5 ")],
    "Up to what age can I take out dental insurance?": [("DPAM01-E4", "Art. 2 ")],
    "With PrimaCare, can I go straight to a gynaecologist without seeing my family doctor?":
        [("RFGM02-E3", "Art. 6 "), ("RFBA02-E5", "Art. 6 ")],
    "What happens if I keep skipping the OptiMed telemedicine call?": [("RSGM02-E5", "Art. 7 ")],
    "What if I forgot to mention an illness when I applied?": [("CAGA04-E2", "Art. 11 ")],
    "Can Groupe Mutuel raise my premiums next year?": [("CAGA04-E2", "Art. 29 ")],
}
Q = np.stack(embed(list(labelled)))


def rank_of_answer(scores: np.ndarray, targets: list[tuple[str, str]]) -> int:
    """1-based rank of the first chunk that answers the question."""

    for rank, i in enumerate(np.argsort(-scores), start=1):
        if any(store.metadata[i]["source"] == s and store.texts[i].startswith(p) for s, p in targets):
            return rank


variants = {
    "raw cosine": None,
    "centred": {},
    "all-but-the-top 1": {"remove": 1},
    "all-but-the-top 2": {"remove": 2},
    "all-but-the-top 3": {"remove": 3},
    "PCA whitening, 32": {"whiten": 32},
    "PCA whitening, 64": {"whiten": 64},
    "PCA whitening, 128": {"whiten": 128},
}
print(f"{'':20s}   MRR  hit@1  hit@3  rank per question")
for name, kwargs in variants.items():
    Ex, Qx = (E, Q) if kwargs is None else (transform(E, **kwargs), transform(Q, **kwargs))
    Ex = Ex / np.linalg.norm(Ex, axis=1, keepdims=True)
    Qx = Qx / np.linalg.norm(Qx, axis=1, keepdims=True)
    ranks = np.array([rank_of_answer(Ex @ q, targets) for q, targets in zip(Qx, labelled.values())])
    print(f"{name:20s} {np.mean(1 / ranks):.3f}  {np.mean(ranks == 1):5.2f}  {np.mean(ranks <= 3):5.2f}  {ranks.tolist()}")

# Centring removes the anisotropy (the mean cosine drops to about 0), but retrieval does not improve here:
# - the statistics come from ~100 chunks, far too few to estimate 1536 dimensions (fit them on a large corpus);
# - questions are short and differ from chunks, so the chunk mean does not centre them, and the removed
#   "common" directions can carry what the question is about;
# - text-embedding-3 is already trained contrastively, unlike the older models where all-but-the-top helped most.
# With 14 questions, a one-rank change moves the MRR a lot: treat any difference here as noise.

In [ ]:
# Extension 4: knowing when there is nothing to find
calibration = {
    question: True,
    "How much does Optimum pay towards glasses for an adult?": True,
    "Does Optimum contribute to a fitness centre subscription?": True,
    unanswerable: False,
    "Is my dog's veterinary treatment covered?": False,
    "What interest does a Groupe Mutuel savings account pay?": False,
}
test = {
    "Are travel vaccines reimbursed under Optimum?": True,
    "What counts as an emergency under PrimaCare?": True,
    "Does Groupe Mutuel insure my car against hail damage?": False,
    "Can I get a mortgage from Groupe Mutuel?": False,
}


def top_scores(questions) -> list[float]:
    return [store.query(v, k=1)[0]["score"] for v in embed(list(questions))]


cal = list(zip(top_scores(calibration), calibration.values(), calibration))
for score, ok, q in sorted(cal):
    print(f"{score:.4f}  {'answerable' if ok else 'unanswerable':12s}  {q}")

lowest_ok = min(s for s, ok, _ in cal if ok)
highest_not = max(s for s, ok, _ in cal if not ok)
threshold = (lowest_ok + highest_not) / 2
print(f"\nThreshold: {threshold:.4f}" + ("" if lowest_ok > highest_not else "  (no clean separation)"))

for score, ok, q in zip(top_scores(test), test.values(), test):
    print(f"{score:.4f}  predicted {'answerable' if score >= threshold else 'unanswerable':12s}  truth {ok}  {q}")

# The alternative: let the model say so.
for q in test:
    prompt = f"""Answer the question using only the policy extracts below.
If the extracts do not answer it, reply exactly: Not covered by these documents.

POLICY EXTRACTS
{format_context(retrieve(q))}

QUESTION
{q}
"""
    print(f"\n{q}\n  -> {client.responses.create(model=MODEL, input=prompt).output_text}")

In [ ]:
# Extension 5: hybrid retrieval
import math
import re
from collections import Counter


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25_docs = [Counter(tokenize(t)) for t in store.texts]
bm25_lengths = np.array([sum(d.values()) for d in bm25_docs])
document_frequency = Counter(term for d in bm25_docs for term in d)
idf = {t: math.log(1 + (len(bm25_docs) - f + 0.5) / (f + 0.5)) for t, f in document_frequency.items()}


def bm25_ranking(query: str, k1: float = 1.5, b: float = 0.75) -> list[int]:
    """Indices of all chunks, best BM25 score first."""

    norm = k1 * (1 - b + b * bm25_lengths / bm25_lengths.mean())
    scores = np.zeros(len(bm25_docs))
    for term in set(tokenize(query)):
        tf = np.array([d[term] for d in bm25_docs])
        scores += idf.get(term, 0) * tf * (k1 + 1) / (tf + norm)
    return [int(i) for i in np.argsort(-scores)]


def dense_ranking(query: str) -> list[int]:
    return [r["index"] for r in retrieve(query, k=len(store))]


def rrf(*rankings: list[int], k: int = 60) -> list[int]:
    """Reciprocal Rank Fusion: each ranking adds 1 / (k + rank) to a chunk's score."""

    fused = Counter()
    for ranking in rankings:
        for rank, i in enumerate(ranking, start=1):
            fused[i] += 1 / (k + rank)
    return [i for i, _ in fused.most_common()]


def label(i: int) -> str:
    return f"{store.metadata[i]['source']} #{store.metadata[i]['position']}"


for q in [
    "SVK",                                                  # abbreviations: BM25 should win
    "What does the LPD/DSG say?",
    "How do I get my money back after paying a doctor?",    # paraphrase: dense should win
    messy,
]:
    dense, sparse = dense_ranking(q), bm25_ranking(q)
    print(f"\n{q}")
    for name, ranking in [("dense", dense), ("BM25", sparse), ("RRF", rrf(dense, sparse))]:
        print(f"  {name:6s} {[label(i) for i in ranking[:3]]}")

# Read a chunk with print(store.texts[i]) to judge which ranking got it right.